# Import libraries


In [1]:
import pandas as pd
import psycopg2
import os
from sqlalchemy import create_engine, Table, MetaData, Column, Integer, String, DateTime, ForeignKey
from sqlalchemy.orm import sessionmaker, relationship, declarative_base, clear_mappers
from dotenv import load_dotenv

load_dotenv()

True

# Connect to database


## Get database information


In [2]:
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
postgres_host = os.getenv("POSTGRES_HOST")
postgres_port = os.getenv("POSTGRES_PORT")
postgres_db = os.getenv("POSTGRES_DB")

## Create connection to database

In [3]:
try:
    conn = psycopg2.connect(
        database=postgres_db,
        user=postgres_user,
        host=postgres_host,
        password=postgres_password,
        port=postgres_port,
    )
    print("Opened database successfully")
except Exception as e:
    print(f"Connection failed: {e}")

Opened database successfully


In [4]:
# Create engine
engine = create_engine(
    f"postgresql://{postgres_user}:{postgres_password}@{postgres_host}:{postgres_port}/{postgres_db}"
)

# Create a session
session = sessionmaker(bind=engine)()

# Create a MetaData instance
metadata = MetaData()

# Reflect the table
company_table = Table("company", metadata, autoload_with=engine)
news_table = Table("news", metadata, autoload_with=engine)
sentiment_table = Table("sentiment", metadata, autoload_with=engine)
statement_table = Table("financialstatement", metadata, autoload_with=engine)

# Define table

In [21]:
Base = declarative_base()
clear_mappers()

## Define company table

In [22]:
class Company(Base):
    __tablename__ = 'company'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    symbol = Column(String)
    name = Column(String)
    circulatingStockVolume = Column(Integer)
    listedStockVolume = Column(Integer)
    marketCap = Column(Integer)
    charterCapital = Column(Integer)
    news_relation = relationship("News", back_populates="company_relation")
    stock_relation = relationship("Stock", back_populates="company_relation")
    statement_relation = relationship("Statement", back_populates="company_relation")


## Define statement table

In [23]:
class Statement(Base):
    __tablename__ = 'financialstatement'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    year = Column(Integer)
    company = Column(Integer, ForeignKey('company.id'))
    
    # Define the relationship to Company
    company_relation = relationship("Company", back_populates="statement_relation")

# Define news table

In [24]:
class News(Base):
    __tablename__ = 'news'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    title = Column(String)
    rawContent = Column(String)
    publishedAt = Column(DateTime)
    company = Column(Integer, ForeignKey('company.id'))
    
    # Define the relationship to Company
    company_relation = relationship("Company", back_populates="news_relation")

## Define stock table

In [25]:
class Stock(Base):
    __tablename__ = 'stock'
    __table_args__ = {'extend_existing': True}
    id = Column(Integer, primary_key=True)
    openingPrice = Column(Integer)
    highestPrice = Column(Integer)
    lowestPrice = Column(Integer)
    closingPrice = Column(Integer)
    date = Column(DateTime)
    company = Column(Integer, ForeignKey('company.id'))
    
    # Define the relationship to Company
    company_relation = relationship("Company", back_populates="stock_relation")

# Get records from database

## Get company records

In [18]:
def get_company_records():
    results = session.query(Company.symbol, Company.circulatingStockVolume, Company.listedStockVolume, Company.marketCap, Company.charterCapital).all()

    # Convert the results to a DataFrame
    df = pd.DataFrame(results, columns=["công ty", "klcp lưu hành", "klcp niêm yết", "vốn thị trường", "vốn điều lệ"])

    return df

## Get financial statement records

In [19]:
def get_statement_records():
    results = session.query(Statement.year, Company.symbol).join(Company).all()
    
    # Chuyển đổi kết quả thành DataFrame
    df = pd.DataFrame(results, columns=['year', 'company_symbol'])

    return df


In [26]:
get_statement_records()
# get_company_records()

,year,company_symbol
0,2019,CAG
1,2018,CAG
2,2017,CAG
3,2016,CAG
4,2023,CAN
...,...,...
16638,2007,CAD
16639,2006,CAD
16640,2023,CAG
16641,2021,CAG
